In [2]:
# Cell 1 — Imports
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score


In [3]:
# Cell 2 — Load training + test datasets (edit file paths)
# X_train contains features, y_train contains the target return
# X_test contains the unseen data for Kaggle submission

X_train_path = r"C:\Users\ryanb\OneDrive\Desktop\ENS Data Camp\X_train_9xQjqvZ.csv"
y_train_path = r"C:\Users\ryanb\OneDrive\Desktop\ENS Data Camp\y_train_Ppwhaz8.csv"
X_test_path  = r"C:\Users\ryanb\Downloads\X_test_1zTtEnD.csv"

x_train = pd.read_csv(X_train_path)
y_train = pd.read_csv(y_train_path)
x_test  = pd.read_csv(X_test_path)

# Merge train features and target into one DataFrame (aligned by ROW_ID)
train = x_train.merge(y_train, on="ROW_ID", how="left")

# Convert regression-style return into binary classification label:
# 1 = positive next-day return, 0 = negative or zero return
train["label"] = (train["target"] > 0).astype(int)


In [7]:
train.head(5)


,ROW_ID,TS,ALLOCATION,RET_20,RET_19,RET_18,RET_17,RET_16,RET_15,RET_14,...,SIGNED_VOLUME_6,SIGNED_VOLUME_5,SIGNED_VOLUME_4,SIGNED_VOLUME_3,SIGNED_VOLUME_2,SIGNED_VOLUME_1,MEDIAN_DAILY_TURNOVER,GROUP,target,label
0,0,DATE_0001,ALLOCATION_01,-0.018192,-0.000306,-0.006881,-0.002393,0.000507,-0.001270,-0.002539,...,0.714129,-0.323847,0.525097,0.363601,-0.219328,NaN,0.096905,1,0.009210,1
1,1,DATE_0001,ALLOCATION_02,-0.006394,-0.001059,0.001565,0.000033,0.002829,0.001725,0.000875,...,-0.896826,-0.636931,-1.074450,-0.748884,-0.718912,NaN,0.009974,4,0.000405,1
2,2,DATE_0001,ALLOCATION_03,-0.016587,-0.004517,-0.005306,0.004314,0.006471,-0.005868,-0.005030,...,1.588321,0.790039,1.394445,0.493521,0.268094,NaN,0.044186,1,0.005967,1
3,3,DATE_0001,ALLOCATION_04,-0.005344,0.002790,0.006937,-0.004246,-0.005051,-0.000330,-0.000117,...,-0.631710,-0.663300,-1.615905,-0.959046,-0.478789,NaN,0.001150,2,-0.000542,0
4,4,DATE_0001,ALLOCATION_05,-0.010506,-0.005491,0.007752,-0.012299,0.002191,0.003282,0.000495,...,1.524626,1.565541,1.563963,1.063209,0.921333,NaN,NaN,4,-0.002579,0


In [ ]:
# Cell 3 — Identify feature columns
# The dataset provides 20-day sequences of:
# - RET_1 ... RET_20
# - SIGNED_VOLUME_1 ... SIGNED_VOLUME_20

ret_cols = [c for c in train.columns if c.startswith("RET_")]
vol_cols = [c for c in train.columns if c.startswith("SIGNED_VOLUME_")]

# Sort columns so the sequence order is consistent across rows
ret_cols = sorted(ret_cols, key=lambda c: int(c.split("_")[1]), reverse=True)
                            # sorting the ret columns by splitting ret_1 for example to ret, "1" 
                            # then converting 1 from string to integer
                            # reverse=True to sort from 20 to 1 not from 1 to 20
vol_cols = sorted(vol_cols, key=lambda c: int(c.split("_")[-1]), reverse=True)
                            


# Ensure all features are numeric and handle missing values safely
for df in [train, x_test]:
    df[ret_cols] = df[ret_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)
    df[vol_cols] = df[vol_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)
    # errors="coerce" means “If conversion fails, replace with NaN instead of crashing.”

    # GROUP defines allocation family → treated as categorical later
    df["GROUP"] = pd.to_numeric(df["GROUP"], errors="coerce").fillna(-1).astype(int)

    # Turnover is a liquidity-related feature
    df["MEDIAN_DAILY_TURNOVER"] = pd.to_numeric(df["MEDIAN_DAILY_TURNOVER"], errors="coerce")

# Fill missing turnover with the median from the training set (robust choice)
turnover_median = train["MEDIAN_DAILY_TURNOVER"].median()
# here we use turnover median only for the training dataset because if we use the testing dataset
# we would be giving hints and hence leakage will occur
for df in [train, x_test]:
    df["MEDIAN_DAILY_TURNOVER"] = df["MEDIAN_DAILY_TURNOVER"].fillna(turnover_median)


In [ ]:
# Cell 4 — Outlier clipping helper
# Finance features often contain spikes/outliers
# Winsorization reduces extreme noise while keeping structure


                        #pd.Series refers to one column
                        # so the input to the function is column s
def winsorize_series(s: pd.Series, lower_q=0.01, upper_q=0.99):
    lo = s.quantile(lower_q)
    hi = s.quantile(upper_q)
    return s.clip(lo, hi)


In [ ]:
# Cell 5 — Feature engineering function
# Converts 20-day sequences of returns + volume into strong summary features

def make_features(df: pd.DataFrame, ret_cols, vol_cols) -> pd.DataFrame:

    # Convert sequences into numpy matrices for fast per-row computation
    R = df[ret_cols].to_numpy()   # returns: shape (n, 20)
    V = df[vol_cols].to_numpy()   # signed volume: shape (n, 20)

    absR = np.abs(R)
    absV = np.abs(V)

    feats = pd.DataFrame(index=df.index) #creating empty data frame with same rows as df to fill it with features

    # --- Core statistical return features ---
    feats["mean_ret"] = R.mean(axis=1)                  # average performance
    feats["vol_ret"] = R.std(axis=1, ddof=1)            # volatility (risk)

    # --- Volume flow features ---
    feats["mean_vol"] = V.mean(axis=1)                  # net buying/selling pressure
    feats["vol_vol"] = V.std(axis=1, ddof=1)            # liquidity instability

    # --- Exponential weighted moving average return (recent days matter more) ---
    span = 10
    alpha = 2 / (span + 1)
    weights = np.array([(1 - alpha) ** i for i in range(19, -1, -1)])
    weights /= weights.sum()
    feats["ewma_ret"] = (R * weights).sum(axis=1)

    # --- Distribution shape: skewness + kurtosis ---
    mu = R.mean(axis=1, keepdims=True)
    sd = R.std(axis=1, ddof=1, keepdims=True) + 1e-12 # dd a tiny number to avoid division by zero
    z = (R - mu) / sd # How far from normal is each day?

    feats["skew_ret"] = (z**3).mean(axis=1) # Positive skew → big upside moves
    feats["kurt_ret"] = (z**4).mean(axis=1) - 3.0
# How often do extreme events happen?
# High kurtosis → frequent shocks
# Low kurtosis → calm behavior

    # --- Drawdown risk features ---
    cum = np.cumprod(1 + R, axis=1)
    peak = np.maximum.accumulate(cum, axis=1)
    dd = (cum - peak) / (peak + 1e-12)

    feats["max_drawdown"] = dd.min(axis=1)              # worst drop
    feats["ulcer_index"] = np.sqrt((dd**2).mean(axis=1))

    # --- Momentum signal ---
    feats["momentum"] = R[:, -5:].sum(axis=1) - R[:, :5].sum(axis=1)

    # --- Consistency feature ---
    feats["win_rate"] = (R > 0).mean(axis=1)

    # --- Liquidity proxy ---
    feats["turnover"] = df["MEDIAN_DAILY_TURNOVER"].values

    # Amihud illiquidity: price movement per unit volume
    feats["amihud"] = (absR / (absV + 1e-8)).mean(axis=1)

    # Return-volume correlation (flow-driven signal)
    Rc = R - R.mean(axis=1, keepdims=True)
    Vc = V - V.mean(axis=1, keepdims=True)
    cov = (Rc * Vc).mean(axis=1)

    feats["corr_ret_vol"] = cov / (Rc.std(axis=1) * Vc.std(axis=1) + 1e-12)

    # Keep GROUP as categorical feature
    feats["GROUP"] = df["GROUP"].values

    # --- GROUP-relative normalization ---
    # Makes features comparable across allocation families
    for col in ["mean_ret", "vol_ret", "mean_vol", "vol_vol",
                "amihud", "momentum", "max_drawdown", "win_rate"]:

        feats[f"{col}_grp_rank"] = feats.groupby("GROUP")[col].rank(pct=True)

        grp_mean = feats.groupby("GROUP")[col].transform("mean")
        grp_std = feats.groupby("GROUP")[col].transform("std").replace(0, 1.0)

        feats[f"{col}_grp_z"] = (feats[col] - grp_mean) / grp_std

    # Clip numeric outliers (keeps model stable)
    for c in feats.columns:
        if c != "GROUP":
            feats[c] = winsorize_series(feats[c])

    return feats


In [6]:
# Cell 6 — Build training + testing feature matrices
X_feat = make_features(train, ret_cols, vol_cols)
y = train["label"]

X_test_feat = make_features(x_test, ret_cols, vol_cols)

# Check shapes
X_feat.shape, X_test_feat.shape


((527073, 31), (31870, 31))

In [7]:
# Cell 7 — Cross-validation with CatBoost (GPU)
# CV prevents overfitting to one split and gives reliable performance estimate

cat_features = ["GROUP"]
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_feat, y), start=1):

    X_tr, X_va = X_feat.iloc[tr_idx], X_feat.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    model = CatBoostClassifier(
        iterations=1200,
        depth=6,
        learning_rate=0.03,
        l2_leaf_reg=8,
        loss_function="Logloss",
        eval_metric="Accuracy",
        task_type="GPU",
        devices="0",
        verbose=0
    )

    model.fit(X_tr, y_tr, cat_features=cat_features)

    preds = model.predict(X_va)
    acc = accuracy_score(y_va, preds)

    cv_scores.append(acc)
    print(f"Fold {fold} Accuracy: {acc:.6f}")

print("\nMean CV Accuracy:", np.mean(cv_scores))
print("Std CV Accuracy:", np.std(cv_scores))


Fold 1 Accuracy: 0.530010
Fold 2 Accuracy: 0.531433
Fold 3 Accuracy: 0.527306
Fold 4 Accuracy: 0.533895
Fold 5 Accuracy: 0.530281

Mean CV Accuracy: 0.5305849531386169
Std CV Accuracy: 0.002137633104627351


In [8]:
# Cell 8 — Train final model on full dataset
# After validation, we retrain using ALL available training data

final_model = CatBoostClassifier(
    iterations=1200,
    depth=6,
    learning_rate=0.03,
    l2_leaf_reg=8,
    loss_function="Logloss",
    task_type="GPU",
    devices="0",
    verbose=200
)

final_model.fit(X_feat, y, cat_features=cat_features)


0:	learn: 0.6930745	total: 75.2ms	remaining: 1m 30s
200:	learn: 0.6894465	total: 10.9s	remaining: 54.2s
400:	learn: 0.6877148	total: 21.8s	remaining: 43.4s
600:	learn: 0.6861889	total: 32.4s	remaining: 32.3s
800:	learn: 0.6847913	total: 41.5s	remaining: 20.7s
1000:	learn: 0.6835003	total: 49.4s	remaining: 9.81s
1199:	learn: 0.6822357	total: 60s	remaining: 0us


In [9]:
# Cell 9 — Predict on test + generate Kaggle submission

# Predict probability of positive return
proba = final_model.predict_proba(X_test_feat)[:, 1]

# Convert probability into binary label using 0.5 threshold
pred_label = (proba >= 0.5).astype(int)

# Submission format required by Kaggle
submission = pd.DataFrame({
    "ROW_ID": x_test["ROW_ID"],
    "label": pred_label
})

submission.to_csv("submission.csv", index=False)

print("Saved submission.csv ✅")
submission.head()


Saved submission.csv ✅


,ROW_ID,label
0,527073,0
1,527074,1
2,527075,0
3,527076,0
4,527077,0
